In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date

In [2]:
from pyspark.sql.functions import current_timestamp, lit, expr, to_date

In [3]:
spark = get_sparkSession("Load gold.dim_date")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
start_date = '20250101'
numYears = 1
_schema_name = "gold"
_table_name = "dim_date"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "init_dim_date.ipynb"

print(f"SPARK-APP : Generating Dates in {_fullname} for {numYears} years starting {start_date}")

SPARK-APP : Generating Dates in gold.dim_date for 1 years starting 20250101


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Defining schema & creating dataframe

_schema = ["date_sk", "date", "year", "quarter", "month", "day", "week_of_year", "day_of_week", "day_name", "month_name", "is_weekend"]

_data = generate_date(start_date,numYears)

df = spark.createDataFrame(data = _data, schema = _schema)

loaded_count = df.count()

In [9]:
df.show(10)
df.printSchema()
print(f"SPARK-APP : Rows entered : {loaded_count}")

+--------+----------+----+-------+-----+---+------------+-----------+---------+----------+----------+
| date_sk|      date|year|quarter|month|day|week_of_year|day_of_week| day_name|month_name|is_weekend|
+--------+----------+----+-------+-----+---+------------+-----------+---------+----------+----------+
|20250101|2025-01-01|2025|      1|   01| 01|          00|          3|Wednesday|   January|     false|
|20250102|2025-01-02|2025|      1|   01| 02|          00|          4| Thursday|   January|     false|
|20250103|2025-01-03|2025|      1|   01| 03|          00|          5|   Friday|   January|     false|
|20250104|2025-01-04|2025|      1|   01| 04|          00|          6| Saturday|   January|      true|
|20250105|2025-01-05|2025|      1|   01| 05|          01|          0|   Sunday|   January|      true|
|20250106|2025-01-06|2025|      1|   01| 06|          01|          1|   Monday|   January|     false|
|20250107|2025-01-07|2025|      1|   01| 07|          01|          2|  Tuesday|   

In [10]:
#Format the data to compatable with gold.dim_date

df = df.withColumn("date_sk", expr("cast(date_sk as int)")) \
       .withColumn("date", to_date("date", "yyyy-MM-dd")) \
       .withColumn("year", expr("cast(year as int)")) \
       .withColumn("quarter", expr("cast(quarter as int)")) \
       .withColumn("month", expr("cast(month as int)")) \
       .withColumn("day", expr("cast(day as int)")) \
       .withColumn("week_of_year", expr("cast(week_of_year as int)")) \
       .withColumn("day_of_week", expr("cast(day_of_week as int)"))

df.printSchema()

root
 |-- date_sk: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- month_name: string (nullable = true)
 |-- is_weekend: boolean (nullable = true)



In [11]:

df_ld = df.withColumn("created_on", current_timestamp()) \
       .withColumn("created_by", lit("utilities - manual load")) \
       .withColumn("modified_on", current_timestamp()) \
       .withColumn("modified_by", lit("utilities - manual load"))

In [12]:
df_ld.show(10)

+--------+----------+----+-------+-----+---+------------+-----------+---------+----------+----------+--------------------+--------------------+--------------------+--------------------+
| date_sk|      date|year|quarter|month|day|week_of_year|day_of_week| day_name|month_name|is_weekend|          created_on|          created_by|         modified_on|         modified_by|
+--------+----------+----+-------+-----+---+------------+-----------+---------+----------+----------+--------------------+--------------------+--------------------+--------------------+
|20250101|2025-01-01|2025|      1|    1|  1|           0|          3|Wednesday|   January|     false|2026-01-28 23:54:...|utilities - manua...|2026-01-28 23:54:...|utilities - manua...|
|20250102|2025-01-02|2025|      1|    1|  2|           0|          4| Thursday|   January|     false|2026-01-28 23:54:...|utilities - manua...|2026-01-28 23:54:...|utilities - manua...|
|20250103|2025-01-03|2025|      1|    1|  3|           0|          5| 

In [13]:
max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)

_data = []
if max_timestamp != '1900-01-01 00:00:00.000000':
    df_ld.write \
      .format("delta") \
      .mode("append") \
      .saveAsTable(_fullname)
    
    _data.append([_schema_name, _schema_name, _table_name, "delta-load", "append", str(datetime.now()), "success", loaded_count, _script_name, _script_name])
    insert_control_logs(spark, _data)
    
else:
    df_ld.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(_fullname)
    
    _data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])
    insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to gold.dim_date and load_controller


In [14]:
spark.sql("select * from gold.dim_date").show(10)

+--------+----------+----+-------+-----+---+------------+-----------+---------+----------+----------+--------------------+--------------------+--------------------+--------------------+
| date_sk|      date|year|quarter|month|day|week_of_year|day_of_week| day_name|month_name|is_weekend|          created_on|          created_by|         modified_on|         modified_by|
+--------+----------+----+-------+-----+---+------------+-----------+---------+----------+----------+--------------------+--------------------+--------------------+--------------------+
|20250911|2025-09-11|2025|      3|    9| 11|          36|          4| Thursday| September|     false|2026-01-28 23:54:...|utilities - manua...|2026-01-28 23:54:...|utilities - manua...|
|20250912|2025-09-12|2025|      3|    9| 12|          36|          5|   Friday| September|     false|2026-01-28 23:54:...|utilities - manua...|2026-01-28 23:54:...|utilities - manua...|
|20250913|2025-09-13|2025|      3|    9| 13|          36|          6| 

In [16]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+-----+-----------+----------+---------+----------+--------------------+-----------+------------+--------------------+-------------------+--------------------+-------------------+
|layer|schema_name|table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|         created_by|         modified_on|        modified_by|
+-----+-----------+----------+---------+----------+--------------------+-----------+------------+--------------------+-------------------+--------------------+-------------------+
| gold|       gold|  dim_date|full-load| overwrite|2026-01-28 23:54:...|    success|         365|2026-01-28 23:54:...|init_dim_date.ipynb|2026-01-28 23:54:...|init_dim_date.ipynb|
+-----+-----------+----------+---------+----------+--------------------+-----------+------------+--------------------+-------------------+--------------------+-------------------+



In [17]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")##"bronze.dim_date_bronze")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|2      |365          |
+-------+-------------+



In [ ]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

In [18]:
spark.stop()